# Partie 1

In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro, ttest_ind, norm, f_oneway, kruskal

from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.multicomp import pairwise_tukeyhsd

sns.set_style("whitegrid")


In [2]:
df_ab = pd.read_csv('ab_test_data.csv')
print(f"\nDonnées chargées : {len(df_ab)} lignes")
print(f"Colonnes : {list(df_ab.columns)}")
print(f"\nAperçu des données :")
print(df_ab.head(10))
print(f"\nTypes de données :")
print(df_ab.dtypes)


Données chargées : 5000 lignes
Colonnes : ['user_id', 'variant', 'converted', 'purchase_amount']

Aperçu des données :
   user_id variant  converted  purchase_amount
0        0       A          0         0.000000
1        1       A          1        50.433272
2        2       A          0         0.000000
3        3       A          0         0.000000
4        4       A          0         0.000000
5        5       A          0         0.000000
6        6       A          0         0.000000
7        7       A          0         0.000000
8        8       A          0         0.000000
9        9       A          0         0.000000

Types de données :
user_id              int64
variant                str
converted            int64
purchase_amount    float64
dtype: object


### Q1.1 Taux de conversion par variante

In [4]:
stats_ab = df_ab.groupby('variant')['converted'].agg(['sum', 'count', 'mean'])
stats_ab.columns = ['conversions', 'visiteurs', 'taux_conversion']

print(stats_ab)

n_A = stats_ab.loc['A', 'visiteurs']
n_B = stats_ab.loc['B', 'visiteurs']
conv_A = stats_ab.loc['A', 'conversions']
conv_B = stats_ab.loc['B', 'conversions']
rate_A = stats_ab.loc['A', 'taux_conversion']
rate_B = stats_ab.loc['B', 'taux_conversion']

         conversions  visiteurs  taux_conversion
variant                                         
A                294       2500           0.1176
B                337       2500           0.1348


In [5]:
diff_absolue = rate_B - rate_A
diff_relative = (rate_B - rate_A) / rate_A * 100

print(f"\nTaux de conversion Variante A : {rate_A:.4f} ({rate_A*100:.2f}%)")
print(f"Taux de conversion Variante B : {rate_B:.4f} ({rate_B*100:.2f}%)")
print(f"\nDifférence absolue : {diff_absolue:.4f} ({diff_absolue*100:.2f} points de pourcentage)")
print(f"Différence relative (lift) : +{diff_relative:.2f}%")


Taux de conversion Variante A : 0.1176 (11.76%)
Taux de conversion Variante B : 0.1348 (13.48%)

Différence absolue : 0.0172 (1.72 points de pourcentage)
Différence relative (lift) : +14.63%


## Q1.2 Visualisation des taux de conversion avec Intervalles de confiance

In [9]:
ci_A = proportion_confint(conv_A, n_A, alpha=0.05, method='wilson')
ci_B = proportion_confint(conv_B, n_B, alpha=0.05, method='wilson')

print(f"IC 95% Variante A : [{ci_A[0]:.4f}, {ci_A[1]:.4f}]")
print(f"IC 95% Variante B : [{ci_B[0]:.4f}, {ci_B[1]:.4f}]")


# Graphique
fig, ax = plt.subplots(figsize=(8, 6))

variants = ['Variante A\n(Contrôle)', 'Variante B\n(Nouveau design)']
rates = [rate_A, rate_B]
errors_low = [rate_A - ci_A[0], rate_B - ci_B[0]]
errors_high = [ci_A[1] - rate_A, ci_B[1] - rate_B]

colors = ['#3498db', '#e74c3c']
bars = ax.bar(variants, rates, color=colors, width=0.5, edgecolor='black', linewidth=0.8)
ax.errorbar(variants, rates, yerr=[errors_low, errors_high],
            fmt='none', ecolor='black', capsize=8, linewidth=2)

# Annotations
for bar, rate, n_conv, n_vis in zip(bars, rates, [conv_A, conv_B], [n_A, n_B]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
            f'{rate*100:.2f}%\n({int(n_conv)}/{int(n_vis)})',
            ha='center', fontsize=11, fontweight='bold')

ax.set_ylabel('Taux de conversion', fontsize=12)
ax.set_title('Test A/B : Taux de conversion par variante\navec intervalles de confiance à 95%', fontsize=13)
ax.set_ylim(0, max(rates) * 1.4)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))

plt.tight_layout()
plt.savefig('q1_2_barplot_conversion.png', dpi=150, bbox_inches='tight')
plt.close()
print("Graphique sauvegardé : q1_2_barplot_conversion.png")


IC 95% Variante A : [0.1056, 0.1308]
IC 95% Variante B : [0.1220, 0.1487]
Graphique sauvegardé : q1_2_barplot_conversion.png


In [10]:
print("\nFormulation des hypothèses :")
print("  H0 (hypothèse nulle) : pA = pB")
print("     -> Le taux de conversion est le même pour les deux variantes")
print("  H1 (hypothèse alternative) : pA ≠ pB")
print("     -> Les taux de conversion sont différents (test bilatéral)")
print(f"  Seuil de significativité : alpha = 0.05")


Formulation des hypothèses :
  H0 (hypothèse nulle) : pA = pB
     -> Le taux de conversion est le même pour les deux variantes
  H1 (hypothèse alternative) : pA ≠ pB
     -> Les taux de conversion sont différents (test bilatéral)
  Seuil de significativité : alpha = 0.05


In [11]:
count = np.array([conv_B, conv_A])
nobs = np.array([n_B, n_A])

z_stat, p_value = proportions_ztest(count, nobs)

print(f"\nRésultats du test Z de proportion :")
print(f"  Z-statistique : {z_stat:.4f}")
print(f"  P-value : {p_value:.4f}")



Résultats du test Z de proportion :
  Z-statistique : 1.8313
  P-value : 0.0671


## Q1.7 Analyse du montant moyen des achats

In [12]:
purchases_A = df_ab[(df_ab['variant'] == 'A') & (df_ab['converted'] == 1)]['purchase_amount']
purchases_B = df_ab[(df_ab['variant'] == 'B') & (df_ab['converted'] == 1)]['purchase_amount']

print(f"\nStatistiques des montants d'achat :")
print(f"\n  Variante A ({len(purchases_A)} acheteurs) :")
print(f"    Moyenne  : {purchases_A.mean():.2f} euros")
print(f"    Médiane  : {purchases_A.median():.2f} euros")
print(f"    Écart-type : {purchases_A.std():.2f} euros")
print(f"    Min / Max : {purchases_A.min():.2f} / {purchases_A.max():.2f} euros")

print(f"\n  Variante B ({len(purchases_B)} acheteurs) :")
print(f"    Moyenne  : {purchases_B.mean():.2f} euros")
print(f"    Médiane  : {purchases_B.median():.2f} euros")
print(f"    Écart-type : {purchases_B.std():.2f} euros")
print(f"    Min / Max : {purchases_B.min():.2f} / {purchases_B.max():.2f} euros")


Statistiques des montants d'achat :

  Variante A (294 acheteurs) :
    Moyenne  : 99.60 euros
    Médiane  : 91.89 euros
    Écart-type : 64.87 euros
    Min / Max : 1.51 / 296.86 euros

  Variante B (337 acheteurs) :
    Moyenne  : 103.14 euros
    Médiane  : 84.41 euros
    Écart-type : 73.90 euros
    Min / Max : 7.10 / 500.77 euros


In [13]:
stat_A, p_shapiro_A = shapiro(purchases_A)
stat_B, p_shapiro_B = shapiro(purchases_B)
print(f"\n  Test de normalité (Shapiro-Wilk) :")
print(f"    Variante A : p-value = {p_shapiro_A:.4f} -> {'Normal' if p_shapiro_A > 0.05 else 'Non normal'}")
print(f"    Variante B : p-value = {p_shapiro_B:.4f} -> {'Normal' if p_shapiro_B > 0.05 else 'Non normal'}")


  Test de normalité (Shapiro-Wilk) :
    Variante A : p-value = 0.0000 -> Non normal
    Variante B : p-value = 0.0000 -> Non normal


In [14]:
from scipy.stats import levene
stat_lev, p_levene = levene(purchases_A, purchases_B)
print(f"\n  Test d'homogénéité des variances (Levene) :")
print(f"    p-value = {p_levene:.4f} -> {'Variances homogènes' if p_levene > 0.05 else 'Variances non homogènes'}")


  Test d'homogénéité des variances (Levene) :
    p-value = 0.6908 -> Variances homogènes


In [15]:
print(f"\n--- Test t de Student ---")
print(f"  H0 : Le montant moyen d'achat est le même pour A et B (muA = muB)")
print(f"  H1 : Les montants moyens sont différents (muA != muB)")


--- Test t de Student ---
  H0 : Le montant moyen d'achat est le même pour A et B (muA = muB)
  H1 : Les montants moyens sont différents (muA != muB)


In [19]:
t_stat, p_value_t = ttest_ind(purchases_B, purchases_A, equal_var=True)
print(f"  t-statistique : {t_stat:.4f}")
print(f"  P-value : {p_value_t:.4f}")

  t-statistique : 0.6369
  P-value : 0.5244


In [20]:
def cohens_d(group1, group2):
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    return (np.mean(group1) - np.mean(group2)) / pooled_std

d = cohens_d(purchases_B.values, purchases_A.values)
print(f"\n  Taille d'effet (Cohen's d) : {d:.4f}")
if abs(d) < 0.2:
    interpretation_d = "négligeable"
elif abs(d) < 0.5:
    interpretation_d = "petit"
elif abs(d) < 0.8:
    interpretation_d = "moyen"
else:
    interpretation_d = "grand"
print(f"  Interprétation : effet {interpretation_d}")

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme
axes[0].hist(purchases_A, bins=30, alpha=0.6, color='#3498db', label=f'A (n={len(purchases_A)})', edgecolor='white')
axes[0].hist(purchases_B, bins=30, alpha=0.6, color='#e74c3c', label=f'B (n={len(purchases_B)})', edgecolor='white')
axes[0].axvline(purchases_A.mean(), color='#2980b9', linestyle='--', linewidth=2, label=f'Moyenne A = {purchases_A.mean():.1f}')
axes[0].axvline(purchases_B.mean(), color='#c0392b', linestyle='--', linewidth=2, label=f'Moyenne B = {purchases_B.mean():.1f}')
axes[0].set_xlabel('Montant (euros)')
axes[0].set_ylabel('Fréquence')
axes[0].set_title('Distribution des montants d\'achat')
axes[0].legend(fontsize=9)

# Boxplot
data_box = [purchases_A, purchases_B]
bp = axes[1].boxplot(data_box, labels=['Variante A', 'Variante B'],
                     patch_artist=True, widths=0.5)
bp['boxes'][0].set_facecolor('#3498db')
bp['boxes'][1].set_facecolor('#e74c3c')
for box in bp['boxes']:
    box.set_alpha(0.7)
axes[1].set_ylabel('Montant (euros)')
axes[1].set_title(f'Comparaison des montants\n(test t: p={p_value_t:.4f}, Cohen d={d:.3f})')

plt.tight_layout()
plt.savefig('q1_7_montants_achat.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nGraphique sauvegardé : q1_7_montants_achat.png")


  Taille d'effet (Cohen's d) : 0.0508
  Interprétation : effet négligeable

Graphique sauvegardé : q1_7_montants_achat.png


C:\Users\Administrateur\AppData\Local\Temp\ipykernel_15460\4157004148.py:34: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = axes[1].boxplot(data_box, labels=['Variante A', 'Variante B'],


## Partie 2

### Q2.1 Statistique descriptives

In [21]:
df_promo = pd.read_csv('promotion_test_data.csv')

print(f"\nDonnées chargées : {len(df_promo)} lignes")
print(f"Colonnes : {list(df_promo.columns)}")
print(f"Groupes : {df_promo['promotion_type'].unique()}")
print(f"Taille par groupe : {df_promo['promotion_type'].value_counts().to_dict()}")


Données chargées : 1200 lignes
Colonnes : ['user_id', 'promotion_type', 'basket_amount']
Groupes : <ArrowStringArray>
['Control', 'Discount_10', 'Free_Shipping', 'Combo']
Length: 4, dtype: str
Taille par groupe : {'Control': 300, 'Discount_10': 300, 'Free_Shipping': 300, 'Combo': 300}


In [22]:
desc_stats = df_promo.groupby('promotion_type')['basket_amount'].agg([
    'count', 'mean', 'median', 'std', 'min',
    lambda x: x.quantile(0.25),
    lambda x: x.quantile(0.75),
    'max'
])
desc_stats.columns = ['N', 'Moyenne', 'Médiane', 'Écart-type', 'Min', 'Q1', 'Q3', 'Max']

print(desc_stats)

                  N     Moyenne     Médiane  Écart-type        Min         Q1  \
promotion_type                                                                  
Combo           300  103.644339  104.441963   29.104643   9.988340  84.251069   
Control         300   84.270352   84.544294   25.706516   4.223625  66.671232   
Discount_10     300   92.023002   92.746838   26.437093  16.720952  74.925556   
Free_Shipping   300   96.738405   95.985524   26.347421  25.343720  80.003184   

                        Q3         Max  
promotion_type                          
Combo           122.456997  181.676834  
Control         101.657982  158.965636  
Discount_10     110.278742  161.115956  
Free_Shipping   116.603435  190.861060  


In [23]:

# Réordonner les groupes
order = ['Control', 'Discount_10', 'Free_Shipping', 'Combo']
desc_stats = desc_stats.reindex(order)

print("\nTableau des statistiques descriptives :")
print(desc_stats.round(2).to_string())

# Extraire les groupes
groups = {name: group['basket_amount'].values
          for name, group in df_promo.groupby('promotion_type')}

print(f"\nObservations clés :")
print(f"  - Les moyennes augmentent progressivement :")
for name in order:
    print(f"    {name:20s} : {groups[name].mean():.2f} euros")
print(f"  - Écart entre Control et Combo : "
      f"{groups['Combo'].mean() - groups['Control'].mean():.2f} euros "
      f"(+{(groups['Combo'].mean() - groups['Control'].mean()) / groups['Control'].mean() * 100:.1f}%)")
print(f"  - Écarts-types relativement homogènes (25-30 euros)")


Tableau des statistiques descriptives :
                  N  Moyenne  Médiane  Écart-type    Min     Q1      Q3     Max
promotion_type                                                                 
Control         300    84.27    84.54       25.71   4.22  66.67  101.66  158.97
Discount_10     300    92.02    92.75       26.44  16.72  74.93  110.28  161.12
Free_Shipping   300    96.74    95.99       26.35  25.34  80.00  116.60  190.86
Combo           300   103.64   104.44       29.10   9.99  84.25  122.46  181.68

Observations clés :
  - Les moyennes augmentent progressivement :
    Control              : 84.27 euros
    Discount_10          : 92.02 euros
    Free_Shipping        : 96.74 euros
    Combo                : 103.64 euros
  - Écart entre Control et Combo : 19.37 euros (+23.0%)
  - Écarts-types relativement homogènes (25-30 euros)


## Q2.2 Visualisation comparative

In [24]:

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Boxplot
colors_promo = ['#95a5a6', '#3498db', '#2ecc71', '#e74c3c']
bp = axes[0].boxplot(
    [groups[name] for name in order],
    labels=['Control', 'Discount\n-10%', 'Livraison\ngratuite', 'Combo\n-15%+livraison'],
    patch_artist=True,
    widths=0.6
)
for patch, color in zip(bp['boxes'], colors_promo):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Ajouter les moyennes
means = [groups[name].mean() for name in order]
axes[0].scatter(range(1, 5), means, color='red', marker='D', s=80, zorder=5, label='Moyenne')
for i, m in enumerate(means):
    axes[0].annotate(f'{m:.1f}', (i + 1, m), textcoords="offset points",
                     xytext=(15, 5), fontsize=9, fontweight='bold', color='red')

axes[0].set_ylabel('Montant du panier (euros)', fontsize=11)
axes[0].set_title('Distribution des montants par type de promotion', fontsize=12)
axes[0].legend(loc='upper left')

# Violin plot
parts = axes[1].violinplot(
    [groups[name] for name in order],
    positions=range(1, 5),
    showmeans=True,
    showmedians=True
)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(colors_promo[i])
    pc.set_alpha(0.7)

axes[1].set_xticks(range(1, 5))
axes[1].set_xticklabels(['Control', 'Discount\n-10%', 'Livraison\ngratuite', 'Combo\n-15%+livraison'])
axes[1].set_ylabel('Montant du panier (euros)', fontsize=11)
axes[1].set_title('Violin plot - Densité des montants par promotion', fontsize=12)

plt.tight_layout()
plt.savefig('q2_2_visualisation_promotions.png', dpi=150, bbox_inches='tight')
plt.close()
print("Graphique sauvegardé : q2_2_visualisation_promotions.png")

Graphique sauvegardé : q2_2_visualisation_promotions.png


C:\Users\Administrateur\AppData\Local\Temp\ipykernel_15460\3799593015.py:5: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = axes[0].boxplot(


In [28]:
df_promo['group_mean'] = df_promo.groupby('promotion_type')['basket_amount'].transform('mean')
df_promo['residual'] = df_promo['basket_amount'] - df_promo['group_mean']
stat_shapiro, p_shapiro = shapiro(df_promo['residual'])

print(f"    Statistique = {stat_shapiro:.4f}")
print(f"    P-value = {p_shapiro:.4f}")

    Statistique = 0.9994
    P-value = 0.9708


In [29]:
for name in order:
    stat, p = shapiro(groups[name])
    status = "Normal" if p > 0.05 else "Non normal"
    print(f"    {name:20s} : p-value = {p:.4f} -> {status}")

    Control              : p-value = 0.9989 -> Normal
    Discount_10          : p-value = 0.7453 -> Normal
    Free_Shipping        : p-value = 0.5455 -> Normal
    Combo                : p-value = 0.4509 -> Normal


In [30]:
fig, ax = plt.subplots(figsize=(7, 6))
stats.probplot(df_promo['residual'], dist="norm", plot=ax)
ax.set_title('Q-Q Plot des résidus de l\'ANOVA', fontsize=12)
plt.tight_layout()
plt.savefig('q2_3_qqplot_residus.png', dpi=150, bbox_inches='tight')
plt.close()
print("\n  Q-Q Plot sauvegardé : q2_3_qqplot_residus.png")


  Q-Q Plot sauvegardé : q2_3_qqplot_residus.png


In [32]:
groups_list = [groups[name] for name in order]
stat_levene, p_levene = levene(*groups_list)

print(f"  Test de Levene :")
print(f"    Statistique = {stat_levene:.4f}")
print(f"    P-value = {p_levene:.4f}")

  Test de Levene :
    Statistique = 1.5474
    P-value = 0.2005


In [33]:
print("\nHypothèses :")
print("  H0 : mu_Control = mu_Discount = mu_FreeShipping = mu_Combo")
print("       (toutes les moyennes sont égales)")
print("  H1 : Au moins une moyenne diffère des autres")
print(f"  Seuil alpha = 0.05")


Hypothèses :
  H0 : mu_Control = mu_Discount = mu_FreeShipping = mu_Combo
       (toutes les moyennes sont égales)
  H1 : Au moins une moyenne diffère des autres
  Seuil alpha = 0.05


In [37]:
f_stat, p_value = f_oneway(groups['Control'], groups['Discount_10'], groups['Free_Shipping'], groups["Combo"])
print(f_stat)
print(p_value)

27.434902171717592
3.685782699381427e-17


In [38]:
print("\n" + "=" * 80)
print(" Q2.4 : ANOVA à un facteur")
print("=" * 80)

print("\nHypothèses :")
print("  H0 : mu_Control = mu_Discount = mu_FreeShipping = mu_Combo")
print("       (toutes les moyennes sont égales)")
print("  H1 : Au moins une moyenne diffère des autres")
print(f"  Seuil alpha = 0.05")

# Calcul détaillé
all_data = df_promo['basket_amount'].values
mean_global = np.mean(all_data)
N = len(all_data)
k = len(order)

print(f"\n--- Calcul détaillé ---")
print(f"  Nombre de groupes (k) : {k}")
print(f"  Nombre total d'observations (N) : {N}")
print(f"  Moyenne globale : {mean_global:.2f} euros")

# SS_between
ss_between = sum(len(groups[name]) * (np.mean(groups[name]) - mean_global)**2
                 for name in order)
print(f"\n  SS_between (variance entre groupes) :")
for name in order:
    contrib = len(groups[name]) * (np.mean(groups[name]) - mean_global)**2
    print(f"    {name:20s} : {len(groups[name])} x ({np.mean(groups[name]):.2f} - {mean_global:.2f})² = {contrib:.2f}")
print(f"    SS_between = {ss_between:.2f}")

# SS_within
ss_within = sum(np.sum((groups[name] - np.mean(groups[name]))**2)
                for name in order)
print(f"\n  SS_within (variance intra-groupes) :")
for name in order:
    contrib = np.sum((groups[name] - np.mean(groups[name]))**2)
    print(f"    {name:20s} : {contrib:.2f}")
print(f"    SS_within = {ss_within:.2f}")

# SS_total
ss_total = np.sum((all_data - mean_global)**2)
print(f"\n  SS_total = {ss_total:.2f}")
print(f"  Vérification : SS_between + SS_within = {ss_between + ss_within:.2f} (= SS_total)")

# Degrés de liberté
df_between = k - 1
df_within = N - k
print(f"\n  Degrés de liberté :")
print(f"    df_between = k - 1 = {df_between}")
print(f"    df_within  = N - k = {df_within}")

# Carrés moyens
ms_between = ss_between / df_between
ms_within = ss_within / df_within
print(f"\n  Carrés moyens :")
print(f"    MS_between = SS_between / df_between = {ss_between:.2f} / {df_between} = {ms_between:.2f}")
print(f"    MS_within  = SS_within / df_within   = {ss_within:.2f} / {df_within} = {ms_within:.2f}")

# F-statistic
f_stat_manual = ms_between / ms_within
p_value_manual = 1 - stats.f.cdf(f_stat_manual, df_between, df_within)
print(f"\n  F-statistic = MS_between / MS_within = {ms_between:.2f} / {ms_within:.2f} = {f_stat_manual:.4f}")
print(f"  P-value = {p_value_manual:.6f}")



 Q2.4 : ANOVA à un facteur

Hypothèses :
  H0 : mu_Control = mu_Discount = mu_FreeShipping = mu_Combo
       (toutes les moyennes sont égales)
  H1 : Au moins une moyenne diffère des autres
  Seuil alpha = 0.05

--- Calcul détaillé ---
  Nombre de groupes (k) : 4
  Nombre total d'observations (N) : 1200
  Moyenne globale : 94.17 euros

  SS_between (variance entre groupes) :
    Control              : 300 x (84.27 - 94.17)² = 29395.12
    Discount_10          : 300 x (92.02 - 94.17)² = 1381.62
    Free_Shipping        : 300 x (96.74 - 94.17)² = 1980.51
    Combo                : 300 x (103.64 - 94.17)² = 26934.48
    SS_between = 59691.73

  SS_within (variance intra-groupes) :
    Control              : 197586.67
    Discount_10          : 208977.05
    Free_Shipping        : 207561.80
    Combo                : 253277.00
    SS_within = 867402.51

  SS_total = 927094.24
  Vérification : SS_between + SS_within = 927094.24 (= SS_total)

  Degrés de liberté :
    df_between = k - 1 = 3